# Talk2Mind — Train Text, Audio & Video Models (Fast Colab version)

Simplified + optimized vs. the earlier notebook:
- **CMU-MOSEI** -> Text model AND Audio model (no more combining Dataset 2 into audio — cuts audio training data volume and file-walking time).
- **Multimodel_Dataset** -> Video model only.
- Dropped pitch (`pyin`) extraction from audio features — it was ~70x slower than everything else combined and added little for this use case.
- Reduced video frames/resolution and audio duration — smaller inputs, faster CNN/LSTM passes.
- Added `.cache()` to every `tf.data` pipeline — features are extracted once and reused every epoch instead of recomputed from the raw file every time.
- Added an optional `MAX_SAMPLES_PER_SPLIT` cap so you can do a 2-minute smoke test before committing to a full run.

Produces the same 4 output files your FastAPI app expects:
```
text_model_final.h5
audio_model_final.h5
video_model_final.h5
text_vectorizer_vocab.pkl
```


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Dataset paths + sanity check

In [ ]:
DATASET1_DIR = "/content/drive/MyDrive/Talk2Mind_data/CMU-MOSEI"          # -> text + audio
DATASET2_DIR = "/content/drive/MyDrive/Talk2Mind_data/Multimodel_Dataset" # -> video only

SAVE_DIR = "/content/drive/MyDrive/Talk2Mind_data/models"

import os
os.makedirs(SAVE_DIR, exist_ok=True)

for p in [DATASET1_DIR, DATASET2_DIR]:
    assert os.path.isdir(p), f"Not found: {p}"
    print(p, "->", os.listdir(p))

# Local (non-Drive) scratch space for feature caching — MUCH faster than caching to Drive.
CACHE_DIR = "/content/cache"
os.makedirs(CACHE_DIR, exist_ok=True)


## 3. Install extra dependencies

In [ ]:
!pip install -q librosa==0.10.2.post1 soundfile==0.12.1


## 4. Global config

**Speed knobs** (the ones most worth tuning if it's still slow):
- `MAX_SAMPLES_PER_SPLIT`: set to e.g. `300` for a fast smoke test, `None` for the full dataset.
- `BATCH_SIZE`: higher = faster per epoch on GPU, until you hit an OOM.
- `VIDEO_NUM_FRAMES` / `VIDEO_FRAME_SIZE`: the single biggest video-speed lever.
- `EPOCHS_*`: EarlyStopping will usually stop well before the max anyway.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
import librosa
import cv2
import pickle
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

EMOTION_LABELS = ["happy", "sad", "angry", "surprise", "disgust", "fear"]
NUM_CLASSES = len(EMOTION_LABELS)

# ---- speed / size controls ----
MAX_SAMPLES_PER_SPLIT = None   # e.g. 300 for a quick smoke test, None = use everything found

# ---- audio feature settings (reduced vs. the first version) ----
AUDIO_SAMPLE_RATE = 16000
AUDIO_DURATION_SEC = 3.0        # was 4.0
AUDIO_N_MFCC = 40
AUDIO_N_MELS = 40               # was 64
AUDIO_HOP_LENGTH = 512
AUDIO_N_FFT = 1024
AUDIO_TIME_STEPS = int(AUDIO_SAMPLE_RATE * AUDIO_DURATION_SEC / AUDIO_HOP_LENGTH) + 1
AUDIO_FEATURE_DIM = AUDIO_N_MFCC + AUDIO_N_MELS + 1   # mfcc + logmel + rms (pitch/pyin dropped — it was the slowest part by far)

# ---- video feature settings (reduced vs. the first version) ----
VIDEO_NUM_FRAMES = 8            # was 16
VIDEO_FRAME_SIZE = 64           # was 96

# ---- text settings ----
TEXT_VOCAB_SIZE = 20000
TEXT_MAX_LEN = 50
TEXT_EMBED_DIM = 128

# ---- training settings ----
BATCH_SIZE = 16                 # was 8 — bump down if you hit an OOM
EPOCHS_TEXT = 10
EPOCHS_AUDIO = 10
EPOCHS_VIDEO = 8
LEARNING_RATE = 1e-4


## 5. Feature extraction helpers (audio + video)

In [ ]:
def load_and_fix_length(path, sr=AUDIO_SAMPLE_RATE, duration=AUDIO_DURATION_SEC):
    y, _ = librosa.load(path, sr=sr, mono=True)
    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode="constant")
    else:
        y = y[:target_len]
    return y


def extract_audio_features(path):
    """MFCC + log-mel + RMS energy, fixed-length, z-normalized.
    (No pitch/pyin — it was ~70x slower than everything else combined for
    negligible gain on this task; drop this comment block if you want to
    add it back for a slight accuracy trade-off against a lot more time.)"""
    path = path.numpy().decode("utf-8") if hasattr(path, "numpy") else path
    y = load_and_fix_length(path)

    mfcc = librosa.feature.mfcc(y=y, sr=AUDIO_SAMPLE_RATE, n_mfcc=AUDIO_N_MFCC,
                                 n_fft=AUDIO_N_FFT, hop_length=AUDIO_HOP_LENGTH)
    mel = librosa.feature.melspectrogram(y=y, sr=AUDIO_SAMPLE_RATE, n_mels=AUDIO_N_MELS,
                                          n_fft=AUDIO_N_FFT, hop_length=AUDIO_HOP_LENGTH)
    log_mel = librosa.power_to_db(mel)
    rms = librosa.feature.rms(y=y, hop_length=AUDIO_HOP_LENGTH)

    T = min(mfcc.shape[1], log_mel.shape[1], rms.shape[1])
    feat = np.concatenate([mfcc[:, :T], log_mel[:, :T], rms[:, :T]], axis=0).T

    if feat.shape[0] < AUDIO_TIME_STEPS:
        feat = np.concatenate([feat, np.zeros((AUDIO_TIME_STEPS - feat.shape[0], feat.shape[1]))], axis=0)
    else:
        feat = feat[:AUDIO_TIME_STEPS, :]

    mean, std = feat.mean(), feat.std() + 1e-6
    return ((feat - mean) / std).astype(np.float32)


_face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def _crop_largest_face(frame_bgr):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = _face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    if len(faces) == 0:
        return frame_bgr
    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
    return frame_bgr[y:y + h, x:x + w]


def extract_face_sequence(video_path, num_frames=VIDEO_NUM_FRAMES, frame_size=VIDEO_FRAME_SIZE):
    path = video_path.numpy().decode("utf-8") if hasattr(video_path, "numpy") else video_path
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = set(np.round(np.linspace(0, max(total - 1, 0), num_frames)).astype(int).tolist())

    grabbed, frame_i = {}, 0
    while cap.isOpened() and len(grabbed) < len(indices):
        ret, frame = cap.read()
        if not ret:
            break
        if frame_i in indices:
            face = _crop_largest_face(frame)
            face = cv2.resize(face, (frame_size, frame_size))
            face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
            grabbed[frame_i] = face.astype(np.float32) / 255.0
        frame_i += 1
    cap.release()

    ordered = [grabbed[i] for i in sorted(grabbed.keys())]
    if len(ordered) == 0:
        ordered = [np.zeros((frame_size, frame_size, 3), dtype=np.float32)]
    while len(ordered) < num_frames:
        ordered.append(ordered[-1])
    return np.stack(ordered[:num_frames], axis=0)

print("Feature extraction helpers ready.")


## 6. Load Dataset 1 — CMU-MOSEI (drives BOTH the Text model and the Audio model)

In [ ]:
LABELS_DIR = os.path.join(DATASET1_DIR, "Labels")

SPLIT_CSV = {
    "train": os.path.join(LABELS_DIR, "Data_Train_modified.csv"),
    "val":   os.path.join(LABELS_DIR, "Data_Val_modified.csv"),
    "test":  os.path.join(LABELS_DIR, "Data_Test_modified.csv"),
}
SPLIT_AUDIO_DIR = {
    "train": os.path.join(DATASET1_DIR, "Audio_chunk", "Train_modified"),
    "val":   os.path.join(DATASET1_DIR, "Audio_chunk", "Val_modified"),
    "test":  os.path.join(DATASET1_DIR, "Audio_chunk", "Test_modified"),
}

# If auto-detection fails for your actual CSV, hardcode real column names here,
# e.g. {"video_id": "clip_id", "start": "start_time", ...}
MANUAL_COLUMNS = {}


def detect_column(df, candidates):
    for c in candidates:
        for col in df.columns:
            if col.strip().lower() == c:
                return col
    return None


def resolve_columns(df):
    cols = {}
    cols["video_id"] = MANUAL_COLUMNS.get("video_id") or detect_column(df, ["video_id", "videoid", "video", "vid"])
    cols["start"] = MANUAL_COLUMNS.get("start") or detect_column(df, ["start", "start_time", "segment_start"])
    cols["end"] = MANUAL_COLUMNS.get("end") or detect_column(df, ["end", "end_time", "segment_end"])
    cols["text"] = MANUAL_COLUMNS.get("text") or detect_column(df, ["text", "transcript", "sentence"])
    for emo in EMOTION_LABELS:
        cols[emo] = MANUAL_COLUMNS.get(emo) or detect_column(df, [emo])
    missing = [k for k, v in cols.items() if v is None]
    if missing:
        raise ValueError(
            f"Could not auto-detect columns {missing}. Actual columns: {list(df.columns)}. "
            f"Fill in MANUAL_COLUMNS above with the real names and re-run this cell."
        )
    return cols


def load_cmu_mosei_split(split):
    csv_path = SPLIT_CSV[split]
    if not os.path.exists(csv_path):
        print(f"[CMU-MOSEI] no CSV for split '{split}' at {csv_path} — skipping.")
        return pd.DataFrame(columns=["audio_path", "text"] + EMOTION_LABELS)

    df = pd.read_csv(csv_path)
    cols = resolve_columns(df)

    audio_dir = SPLIT_AUDIO_DIR[split]
    rows = []
    for _, row in df.iterrows():
        vid = row[cols["video_id"]]
        start = float(row[cols["start"]])
        end = float(row[cols["end"]])
        fname = f"{vid}_{start:.4f}_{end:.4f}.wav"
        path = os.path.join(audio_dir, fname)
        if os.path.exists(path):
            rec = {"audio_path": path, "text": str(row[cols["text"]])}
            for emo in EMOTION_LABELS:
                rec[emo] = float(row[cols[emo]])
            rows.append(rec)

    out = pd.DataFrame(rows)
    if MAX_SAMPLES_PER_SPLIT is not None and len(out) > MAX_SAMPLES_PER_SPLIT:
        out = out.sample(n=MAX_SAMPLES_PER_SPLIT, random_state=42).reset_index(drop=True)
    print(f"[CMU-MOSEI] split='{split}': {len(out)}/{len(df)} rows matched to an audio file"
          f"{' (capped by MAX_SAMPLES_PER_SPLIT)' if MAX_SAMPLES_PER_SPLIT else ''}.")
    return out


cmu_train_df = load_cmu_mosei_split("train")
cmu_val_df = load_cmu_mosei_split("val")
cmu_test_df = load_cmu_mosei_split("test")

cmu_train_df.head()


## 7. Load Dataset 2 — Video only (RAVDESS-style, emotion coded in filename)

In [ ]:
RAVDESS_EMOTION_CODE_MAP = {
    "01": None, "02": None, "03": "happy", "04": "sad",
    "05": "angry", "06": "fear", "07": "disgust", "08": "surprise",
}

def parse_ravdess_label(filename):
    parts = os.path.splitext(filename)[0].split("-")
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    if len(parts) >= 3:
        mapped = RAVDESS_EMOTION_CODE_MAP.get(parts[2])
        if mapped:
            vec[EMOTION_LABELS.index(mapped)] = 1.0
    return vec


def load_ds2_video():
    root = os.path.join(DATASET2_DIR, "Video_Dataset", "Video_Dataset")
    rows = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith(".mp4"):
                path = os.path.join(dirpath, f)
                label = parse_ravdess_label(f)
                rows.append({"video_path": path, "label": label})
    df = pd.DataFrame(rows)
    print(f"[Dataset2 Video] found {len(df)} .mp4 files under {root}")
    return df


ds2_video_df = load_ds2_video()

def label_distribution(df, label_col):
    if len(df) == 0:
        return
    stacked = np.stack(df[label_col].values)
    for i, emo in enumerate(EMOTION_LABELS):
        print(f"  {emo}: {int(stacked[:, i].sum())}")
    print(f"  neutral/unlabeled (all-zero): {int((stacked.sum(axis=1) == 0).sum())}")

print("Dataset2 Video label distribution:")
label_distribution(ds2_video_df, "label")

ds2_video_train, ds2_video_val = train_test_split(ds2_video_df, test_size=0.15, random_state=42)
if MAX_SAMPLES_PER_SPLIT is not None:
    ds2_video_train = ds2_video_train.sample(n=min(MAX_SAMPLES_PER_SPLIT, len(ds2_video_train)), random_state=42)
    ds2_video_val = ds2_video_val.sample(n=min(MAX_SAMPLES_PER_SPLIT // 5 or 1, len(ds2_video_val)), random_state=42)
print(f"Dataset2 video: {len(ds2_video_train)} train / {len(ds2_video_val)} val")


## 8. Build final (path, label) pair lists

In [ ]:
def cmu_rows_to_pairs(df):
    return [(row["audio_path"], np.array([row[e] for e in EMOTION_LABELS], dtype=np.float32))
            for _, row in df.iterrows()]

def ds2_rows_to_pairs(df, path_col):
    return [(row[path_col], row["label"]) for _, row in df.iterrows()]

# Audio model: CMU-MOSEI only (dropped Dataset 2 audio to cut training time/volume).
audio_train_pairs = cmu_rows_to_pairs(cmu_train_df)
audio_val_pairs = cmu_rows_to_pairs(cmu_val_df)
print(f"AUDIO model (CMU-MOSEI only): {len(audio_train_pairs)} train / {len(audio_val_pairs)} val samples")

# Video model: Dataset 2 (RAVDESS) only.
video_train_pairs = ds2_rows_to_pairs(ds2_video_train, "video_path")
video_val_pairs = ds2_rows_to_pairs(ds2_video_val, "video_path")
print(f"VIDEO model: {len(video_train_pairs)} train / {len(video_val_pairs)} val samples")


## 9. tf.data pipelines (with caching — features are computed once, then reused every epoch)

In [ ]:
def make_audio_tf_dataset(pairs, cache_name, batch_size=BATCH_SIZE, shuffle=True):
    paths = [p for p, _ in pairs]
    labels = np.stack([l for _, l in pairs]) if pairs else np.zeros((0, NUM_CLASSES), dtype=np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def _load(path, label):
        feat = tf.py_function(func=extract_audio_features, inp=[path], Tout=tf.float32)
        feat.set_shape([AUDIO_TIME_STEPS, AUDIO_FEATURE_DIM])
        label.set_shape([NUM_CLASSES])
        return feat, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.cache(os.path.join(CACHE_DIR, cache_name))   # <- extraction happens once, not every epoch
    if shuffle:
        ds = ds.shuffle(buffer_size=max(len(paths), 1), reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def make_video_tf_dataset(pairs, cache_name, batch_size=BATCH_SIZE, shuffle=True):
    paths = [p for p, _ in pairs]
    labels = np.stack([l for _, l in pairs]) if pairs else np.zeros((0, NUM_CLASSES), dtype=np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def _load(path, label):
        frames = tf.py_function(func=extract_face_sequence, inp=[path], Tout=tf.float32)
        frames.set_shape([VIDEO_NUM_FRAMES, VIDEO_FRAME_SIZE, VIDEO_FRAME_SIZE, 3])
        label.set_shape([NUM_CLASSES])
        return frames, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.cache(os.path.join(CACHE_DIR, cache_name))
    if shuffle:
        ds = ds.shuffle(buffer_size=max(len(paths), 1), reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def make_text_tf_dataset(df, vectorizer, batch_size=BATCH_SIZE, shuffle=True):
    texts = df["text"].astype(str).tolist()
    labels = df[EMOTION_LABELS].values.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=max(len(texts), 1), reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda x, y: (vectorizer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

print("tf.data pipeline builders ready.")


## 10. MODEL 1 — Text (Embedding + BiLSTM + ANN), trained on CMU-MOSEI

In [ ]:
def build_text_model():
    inputs = layers.Input(shape=(TEXT_MAX_LEN,), dtype="int32", name="text_tokens")
    x = layers.Embedding(input_dim=TEXT_VOCAB_SIZE, output_dim=TEXT_EMBED_DIM, mask_zero=True)(inputs)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    embedding = layers.Dense(128, activation="relu", name="text_emotion_embedding")(x)
    x = layers.Dropout(0.3)(embedding)
    outputs = layers.Dense(NUM_CLASSES, activation="sigmoid", name="text_output")(x)

    model = Model(inputs, outputs, name="TextEmotionModel")
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model


text_vectorizer = layers.TextVectorization(max_tokens=TEXT_VOCAB_SIZE, output_mode="int",
                                            output_sequence_length=TEXT_MAX_LEN)
text_vectorizer.adapt(cmu_train_df["text"].astype(str).tolist())

text_train_ds = make_text_tf_dataset(cmu_train_df, text_vectorizer, shuffle=True)
text_val_ds = make_text_tf_dataset(cmu_val_df, text_vectorizer, shuffle=False)

text_model = build_text_model()
text_model.summary()


In [ ]:
text_history = text_model.fit(
    text_train_ds, validation_data=text_val_ds,
    epochs=EPOCHS_TEXT,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
)


In [ ]:
text_model_path = os.path.join(SAVE_DIR, "text_model_final.h5")
text_model.save(text_model_path)

vocab_path = os.path.join(SAVE_DIR, "text_vectorizer_vocab.pkl")
pickle.dump(text_vectorizer.get_vocabulary(), open(vocab_path, "wb"))

print("Saved:", text_model_path)
print("Saved:", vocab_path)


## 11. MODEL 2 — Audio (1D-CNN + LSTM), trained on CMU-MOSEI only

In [ ]:
def build_audio_model():
    inputs = layers.Input(shape=(AUDIO_TIME_STEPS, AUDIO_FEATURE_DIM), name="audio_features")

    x = layers.Conv1D(64, 5, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(128, 5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(256, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.LSTM(64)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    embedding = layers.Dense(128, activation="relu", name="speech_emotion_embedding")(x)
    x = layers.Dropout(0.3)(embedding)
    outputs = layers.Dense(NUM_CLASSES, activation="sigmoid", name="audio_output")(x)

    model = Model(inputs, outputs, name="SpeechEmotionModel")
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model


audio_train_ds = make_audio_tf_dataset(audio_train_pairs, "audio_train_cache", shuffle=True)
audio_val_ds = make_audio_tf_dataset(audio_val_pairs, "audio_val_cache", shuffle=False)

audio_model = build_audio_model()
audio_model.summary()


In [ ]:
audio_history = audio_model.fit(
    audio_train_ds, validation_data=audio_val_ds,
    epochs=EPOCHS_AUDIO,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
)


In [ ]:
audio_model_path = os.path.join(SAVE_DIR, "audio_model_final.h5")
audio_model.save(audio_model_path)
print("Saved:", audio_model_path)


## 12. MODEL 3 — Video (TimeDistributed CNN + LSTM), trained on Dataset 2 (RAVDESS)

In [ ]:
def build_video_model():
    frame_input = layers.Input(shape=(VIDEO_FRAME_SIZE, VIDEO_FRAME_SIZE, 3))
    c = layers.Conv2D(32, 3, activation="relu", padding="same")(frame_input)
    c = layers.BatchNormalization()(c)
    c = layers.MaxPooling2D()(c)
    c = layers.Conv2D(64, 3, activation="relu", padding="same")(c)
    c = layers.BatchNormalization()(c)
    c = layers.MaxPooling2D()(c)
    c = layers.Conv2D(128, 3, activation="relu", padding="same")(c)
    c = layers.BatchNormalization()(c)
    c = layers.GlobalAveragePooling2D()(c)
    cnn_backbone = Model(frame_input, c, name="CustomCNNBackbone")

    inputs = layers.Input(shape=(VIDEO_NUM_FRAMES, VIDEO_FRAME_SIZE, VIDEO_FRAME_SIZE, 3), name="video_frames")
    x = layers.TimeDistributed(cnn_backbone)(inputs)
    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.LSTM(64)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    embedding = layers.Dense(128, activation="relu", name="visual_emotion_embedding")(x)
    x = layers.Dropout(0.3)(embedding)
    outputs = layers.Dense(NUM_CLASSES, activation="sigmoid", name="video_output")(x)

    model = Model(inputs, outputs, name="FacialEmotionModel")
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model


video_train_ds = make_video_tf_dataset(video_train_pairs, "video_train_cache", shuffle=True)
video_val_ds = make_video_tf_dataset(video_val_pairs, "video_val_cache", shuffle=False)

video_model = build_video_model()
video_model.summary()


In [ ]:
video_history = video_model.fit(
    video_train_ds, validation_data=video_val_ds,
    epochs=EPOCHS_VIDEO,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
)


In [ ]:
video_model_path = os.path.join(SAVE_DIR, "video_model_final.h5")
video_model.save(video_model_path)
print("Saved:", video_model_path)


## 13. Confirm everything landed on Drive

In [ ]:
print(os.listdir(SAVE_DIR))


## 14. Quick sanity-check inference

In [ ]:
sample_text = ["I've been feeling really overwhelmed and tired lately"]
tokens = text_vectorizer(sample_text)
print("Text prediction:", dict(zip(EMOTION_LABELS, text_model.predict(tokens, verbose=0)[0])))

if len(audio_train_pairs) > 0:
    feat = np.expand_dims(extract_audio_features(audio_train_pairs[0][0]), 0)
    print("Audio prediction:", dict(zip(EMOTION_LABELS, audio_model.predict(feat, verbose=0)[0])))

if len(video_train_pairs) > 0:
    frames = np.expand_dims(extract_face_sequence(video_train_pairs[0][0]), 0)
    print("Video prediction:", dict(zip(EMOTION_LABELS, video_model.predict(frames, verbose=0)[0])))


## Next step / further speed tuning

Drop `text_model_final.h5`, `audio_model_final.h5`, `video_model_final.h5`, and
`text_vectorizer_vocab.pkl` from `SAVE_DIR` into the `models/` folder of the
FastAPI + Streamlit app — no code changes needed there.

**If it's still too slow**, in order of impact:
1. Set `MAX_SAMPLES_PER_SPLIT` (cell 4) to a few hundred first, confirm the whole
   notebook runs end to end, *then* set it to `None` and let the full run go.
2. Lower `VIDEO_NUM_FRAMES` further (e.g. 6) — video is almost always the
   slowest of the three.
3. Make sure Colab's GPU is actually on: `Runtime -> Change runtime type -> GPU`.
   Re-run cell 4 and confirm `GPU: [...]` is non-empty.
4. Raise `BATCH_SIZE` until you hit an out-of-memory error, then back off by one step.
